# 03 错误类型分析与人工复核

本模块使用平衡 nested-CV 的 OOF logits，经 sigmoid 转换后以 0.5 固定阈值定义 false positives (FP) 与 false negatives (FN)。人工试听函数保留，但默认不自动嵌入音频，以免批量执行产生巨大的 notebook 输出。

In [1]:
from pathlib import Path
import os
import sys

def find_project_root(start=None):
    path = Path(start or Path.cwd()).resolve()
    for candidate in [path, *path.parents]:
        if (candidate / "project_config.py").is_file():
            return candidate
    raise FileNotFoundError("Run this notebook from inside the project repository.")

PROJECT_ROOT = find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from project_config import WORKSPACE_DIR

os.environ.setdefault("MPLBACKEND", "Agg")
os.environ.setdefault("MPLCONFIGDIR", str(WORKSPACE_DIR / "matplotlib_cache"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

pd.set_option("display.max_columns", 60)
pd.set_option("display.max_colwidth", 120)

def require_columns(frame, required, name):
    missing = sorted(set(required) - set(frame.columns))
    if missing:
        raise ValueError(f"{name} is missing required columns: {missing}")

def save_figure(fig, stem):
    fig.tight_layout()
    fig.savefig(OUTPUT_DIR / f"{stem}.png", dpi=300, bbox_inches="tight")
    fig.savefig(OUTPUT_DIR / f"{stem}.pdf", bbox_inches="tight")
    plt.close(fig)

def write_latex(frame, name, **kwargs):
    (OUTPUT_DIR / name).write_text(frame.to_latex(index=False, **kwargs), encoding="utf-8")

from scipy.special import expit
import soundfile as sf
from IPython.display import Audio
from project_config import BINARY_LABELS_PATH, CLIPS_DIR

PROBABILITY_THRESHOLD = 0.50
BOOTSTRAP_ITERATIONS = 5000  # 本模块不执行 bootstrap；保留统一论文配置。
RANDOM_SEED = 2021
REPRESENTATIVES_PER_ERROR = 3
ENABLE_AUDIO_PLAYBACK = False
MODEL_ROOT = WORKSPACE_DIR / "models" / "nested_cv_balanced"
OUTPUT_DIR = WORKSPACE_DIR / "paper_analysis" / "03_error_review"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Model root: {MODEL_ROOT}")
print(f"Output directory: {OUTPUT_DIR}")

Model root: /home/zenith/Projects/strom_petrels/workspace/models/nested_cv_balanced
Output directory: /home/zenith/Projects/strom_petrels/workspace/paper_analysis/03_error_review


In [2]:
parts = []
for fold_dir in sorted(MODEL_ROOT.glob("outer_*")):
    path = fold_dir / "predictions.csv"
    if path.is_file():
        part = pd.read_csv(path)
        require_columns(part, {"file", "class_folder", "true_label", "score"}, str(path))
        part["outer_fold"] = fold_dir.name.removeprefix("outer_")
        parts.append(part)
if len(parts) != 5:
    raise FileNotFoundError("Five balanced outer-fold prediction files are required")

pred = pd.concat(parts, ignore_index=True)
pred["raw_logit"] = pd.to_numeric(pred["score"], errors="raise")
pred["probability"] = expit(pred["raw_logit"].to_numpy(float))
pred["true_label"] = pd.to_numeric(pred["true_label"], errors="raise").astype(int)
pred["predicted_label"] = (pred["probability"] >= PROBABILITY_THRESHOLD).astype(int)
pred["file_name"] = pred["file"].astype(str).map(lambda value: Path(value).name)

labels = pd.read_csv(BINARY_LABELS_PATH)
require_columns(labels, {"clip_id", "negative_type", "species", "source_file", "clip_path"}, "binary labels")
labels["file_name"] = labels["clip_id"].astype(str) + ".wav"
pred = pred.merge(labels[["file_name", "clip_id", "negative_type", "species", "source_file", "clip_path"]],
                  on="file_name", how="left", validate="one_to_one")
if len(pred) != 2000 or pred.clip_id.isna().any() or pred.clip_id.nunique() != 2000:
    raise AssertionError("Expected 2,000 unique, metadata-matched OOF predictions")

pred["error_type"] = np.select(
    [(pred.true_label == 0) & (pred.predicted_label == 1),
     (pred.true_label == 1) & (pred.predicted_label == 0)],
    ["false_positive", "false_negative"], default="correct"
)
errors = pred[pred.error_type != "correct"].copy().sort_values(
    ["error_type", "probability"], ascending=[True, False]
)
errors.to_csv(OUTPUT_DIR / "error_manifest.csv", index=False)

error_counts = pred.groupby("error_type").size().rename("clips").reset_index()
error_counts.to_csv(OUTPUT_DIR / "error_counts.csv", index=False)
write_latex(error_counts, "error_summary.tex")
display(error_counts)
display(errors.head(20))

,error_type,clips
0,correct,1991
1,false_negative,3
2,false_positive,6


,file,class_folder,true_label,score,outer_fold,raw_logit,probability,predicted_label,file_name,clip_id,negative_type,species,source_file,clip_path,error_type
1994,/mnt/data2/skokholm/alyzw89/workspace/nested_cv_datasets/outer_05/test/storm_petrel/storm_petrel_00991_042_210522_18...,storm_petrel,1,-1.436692,05,-1.436692,0.192058,0,storm_petrel_00991_042_210522_184504_0031245000.wav,storm_petrel_00991_042_210522_184504_0031245000,NaN,Hydrobates pelagicus_European Storm-Petrel,042_210522_184504.flac,/mnt/data2/skokholm/alyzw89/workspace/clips/storm_petrel/storm_petrel_00991_042_210522_184504_0031245000.wav,false_negative
1167,/mnt/data2/skokholm/alyzw89/workspace/nested_cv_datasets/outer_03/test/storm_petrel/storm_petrel_00981_112_210628_22...,storm_petrel,1,-1.732729,03,-1.732729,0.150239,0,storm_petrel_00981_112_210628_225602_0013761000.wav,storm_petrel_00981_112_210628_225602_0013761000,NaN,Hydrobates pelagicus_European Storm-Petrel,112_210628_225602.flac,/mnt/data2/skokholm/alyzw89/workspace/clips/storm_petrel/storm_petrel_00981_112_210628_225602_0013761000.wav,false_negative
1118,/mnt/data2/skokholm/alyzw89/workspace/nested_cv_datasets/outer_03/test/storm_petrel/storm_petrel_00901_112_210628_22...,storm_petrel,1,-1.754862,03,-1.754862,0.147435,0,storm_petrel_00901_112_210628_225602_0012573000.wav,storm_petrel_00901_112_210628_225602_0012573000,NaN,Hydrobates pelagicus_European Storm-Petrel,112_210628_225602.flac,/mnt/data2/skokholm/alyzw89/workspace/clips/storm_petrel/storm_petrel_00901_112_210628_225602_0012573000.wav,false_negative
39,/mnt/data2/skokholm/alyzw89/workspace/nested_cv_datasets/outer_01/test/not_storm_petrel/environment_noise_01636_042_...,not_storm_petrel,0,1.780659,01,1.780659,0.855778,1,environment_noise_01636_042_210803_211120_0007494000.wav,environment_noise_01636_042_210803_211120_0007494000,environment_noise,Engine_Engine,042_210803_211120.flac,/mnt/data2/skokholm/alyzw89/workspace/clips/not_storm_petrel/environment_noise_01636_042_210803_211120_0007494000.wav,false_positive
1685,/mnt/data2/skokholm/alyzw89/workspace/nested_cv_datasets/outer_05/test/not_storm_petrel/environment_noise_01857_042_...,not_storm_petrel,0,1.364355,05,1.364355,0.796467,1,environment_noise_01857_042_210628_225556_0019977000.wav,environment_noise_01857_042_210628_225556_0019977000,environment_noise,Dog_Dog,042_210628_225556.flac,/mnt/data2/skokholm/alyzw89/workspace/clips/not_storm_petrel/environment_noise_01857_042_210628_225556_0019977000.wav,false_positive
1320,/mnt/data2/skokholm/alyzw89/workspace/nested_cv_datasets/outer_04/test/not_storm_petrel/other_bird_01272_067_210716_...,not_storm_petrel,0,1.302391,04,1.302391,0.786237,1,other_bird_01272_067_210716_213947_0015075000.wav,other_bird_01272_067_210716_213947_0015075000,other_bird,Puffinus puffinus_Manx Shearwater,067_210716_213947.flac,/mnt/data2/skokholm/alyzw89/workspace/clips/not_storm_petrel/other_bird_01272_067_210716_213947_0015075000.wav,false_positive
110,/mnt/data2/skokholm/alyzw89/workspace/nested_cv_datasets/outer_01/test/not_storm_petrel/environment_noise_01966_042_...,not_storm_petrel,0,1.237185,01,1.237185,0.775074,1,environment_noise_01966_042_210803_211120_0019731000.wav,environment_noise_01966_042_210803_211120_0019731000,environment_noise,Dog_Dog,042_210803_211120.flac,/mnt/data2/skokholm/alyzw89/workspace/clips/not_storm_petrel/environment_noise_01966_042_210803_211120_0019731000.wav,false_positive
1700,/mnt/data2/skokholm/alyzw89/workspace/nested_cv_datasets/outer_05/test/not_storm_petrel/environment_noise_01894_011_...,not_storm_petrel,0,0.637547,05,0.637547,0.654199,1,environment_noise_01894_011_210531_221151_0022530000.wav,environment_noise_01894_011_210531_221151_0022530000,environment_noise,Dog_Dog,011_210531_221151.flac,/mnt/data2/skokholm/alyzw89/workspace/clips/not_storm_petrel/environment_noise_01894_011_210531_221151_0022530000.wav,false_positive
1678,/mnt/data2/skokholm/alyzw89/workspace/nested_cv_datasets/outer_05/test/not_storm_petrel/envi

## 错误在元数据维度上的分布

负类亚型主要解释 FP；FN 的 `negative_type` 按“不适用”处理。物种、来源录音与 fold 分布用于定位系统性错误来源，不把窗口数误解为独立生物学重复。

In [3]:
summary_parts = []
for dimension in ["negative_type", "species", "source_file", "outer_fold"]:
    table = (errors.assign(**{dimension: errors[dimension].fillna("not_applicable")})
             .groupby(["error_type", dimension], dropna=False).size()
             .rename("clips").reset_index().rename(columns={dimension: "level"}))
    table.insert(0, "dimension", dimension)
    summary_parts.append(table)
error_breakdown = pd.concat(summary_parts, ignore_index=True)
error_breakdown.to_csv(OUTPUT_DIR / "error_breakdown.csv", index=False)

fig, axes = plt.subplots(2, 2, figsize=(14, 9))
fp = errors[errors.error_type == "false_positive"]
fp.groupby(fp.negative_type.fillna("unknown")).size().sort_values().plot.barh(ax=axes[0, 0], color="#d95f02")
axes[0, 0].set(title="False positives by negative subtype", xlabel="Clips", ylabel="Negative subtype")

species_counts = errors.groupby(["species", "error_type"]).size().unstack(fill_value=0)
species_counts.loc[species_counts.sum(axis=1).nlargest(10).index].plot.barh(ax=axes[0, 1])
axes[0, 1].set(title="Errors by annotated species", xlabel="Clips", ylabel="Species")

source_counts = errors.groupby(["source_file", "error_type"]).size().unstack(fill_value=0)
source_counts.loc[source_counts.sum(axis=1).nlargest(10).index].plot.barh(ax=axes[1, 0])
axes[1, 0].set(title="Errors by source recording", xlabel="Clips", ylabel="Source recording")

errors.groupby(["outer_fold", "error_type"]).size().unstack(fill_value=0).plot.bar(ax=axes[1, 1])
axes[1, 1].set(title="Errors by outer fold", xlabel="Outer fold", ylabel="Clips")
axes[1, 1].tick_params(axis="x", rotation=0)
save_figure(fig, "error_type_distributions")
display(error_breakdown.head(30))

,dimension,error_type,level,clips
0,negative_type,false_negative,not_applicable,3
1,negative_type,false_positive,environment_noise,5
2,negative_type,false_positive,other_bird,1
3,species,false_negative,Hydrobates pelagicus_European Storm-Petrel,3
4,species,false_positive,Dog_Dog,4
5,species,false_positive,Engine_Engine,1
6,species,false_positive,Puffinus puffinus_Manx Shearwater,1
7,source_file,false_negative,042_210522_184504.flac,1
8,source_file,false_negative,112_210628_225602.flac,2
9,source_file,false_positive,011_210531_221151.flac,1


## 代表性错误频谱图与人工试听接口

代表样本按置信程度选择：FP 取最高 sigmoid 分数，FN 取最低分数。频谱图用于人工定位声学模式，不作为独立的统计检验。

In [4]:
def resolve_clip(row):
    candidates = [Path(str(row["clip_path"])), Path(str(row["file"]))]
    if row["true_label"] == 1:
        candidates.append(CLIPS_DIR / "storm_petrel" / row["file_name"])
    else:
        candidates.extend([
            CLIPS_DIR / "not_storm_petrel" / str(row["negative_type"]) / row["file_name"],
            CLIPS_DIR / "not_storm_petrel" / row["file_name"],
        ])
    for candidate in candidates:
        if candidate.is_file():
            return candidate
    matches = list(CLIPS_DIR.rglob(row["file_name"]))
    if matches:
        return matches[0]
    raise FileNotFoundError(f"Cannot resolve audio for {row['file_name']}")

fp_representatives = errors[errors.error_type == "false_positive"].nlargest(REPRESENTATIVES_PER_ERROR, "probability")
fn_representatives = errors[errors.error_type == "false_negative"].nsmallest(REPRESENTATIVES_PER_ERROR, "probability")
representatives = pd.concat([fp_representatives, fn_representatives], ignore_index=True)

fig, axes = plt.subplots(max(len(representatives), 1), 1, figsize=(11, 2.4 * max(len(representatives), 1)), squeeze=False)
for ax, (_, row) in zip(axes.ravel(), representatives.iterrows()):
    path = resolve_clip(row)
    audio, sample_rate = sf.read(path)
    if getattr(audio, "ndim", 1) > 1:
        audio = audio.mean(axis=1)
    ax.specgram(audio, Fs=sample_rate, NFFT=1024, noverlap=512, cmap="magma")
    ax.set(title=f"{row['error_type']} | score={row['probability']:.3f} | {row['file_name']}",
           xlabel="Time (s)", ylabel="Frequency (Hz)")
save_figure(fig, "representative_error_spectrograms")

representatives.to_csv(OUTPUT_DIR / "representative_errors.csv", index=False)

def listen_to_error(row):
    """在交互式复核时显示元数据并返回音频播放器。"""
    path = resolve_clip(row)
    display(row[["error_type", "outer_fold", "probability", "negative_type", "species", "source_file"]].to_frame().T)
    return Audio(filename=str(path))

if ENABLE_AUDIO_PLAYBACK:
    for _, row in representatives.iterrows():
        display(listen_to_error(row))
else:
    print("Audio playback is disabled for batch execution; set ENABLE_AUDIO_PLAYBACK=True for manual review.")

Audio playback is disabled for batch execution; set ENABLE_AUDIO_PLAYBACK=True for manual review.
